In [ ]:
# PARTE A.1  División train/test y tabla de diferencia de medias
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from scipy import stats

# en este paso fijo paths y cargo la base
PATH_BASES = Path.cwd()
file_resp = PATH_BASES / "respondieron_final.xlsx"
df = pd.read_excel(file_resp)

# en este paso defino lista candidata de variables limpias de TPs previos
# reemplazar o ampliar según las columnas reales que tenga la base
candidatas = [
    "edad",                    # num
    "sexo",                    # cat
    "educacion",               # cat
    "nivel_educativo",         # cat
    "region",                  # cat
    "provincia",               # cat
    "ocupado",                 # cat/dummy
    "categoria_ocupacional",   # cat
    "estado_civil",            # cat
    "cantidad_miembros",       # num
    "tamanio_hogar",           # num
    "ad_equiv_hogar",          # num
    "hogar_nro",               # num si sirve como tamaño
    "tipo_vivienda"            # cat
]

# en este paso excluyo todo lo ligado a ingreso
bloquear = {"ingreso", "itf", "itfipc", "itf_hogar", "ingreso_per_capita", "ingreso_necesario"}
def es_ingreso(col):
    c = str(col).lower()
    return any(tok in c for tok in bloquear)

# en este paso selecciono variables disponibles y válidas
vars_X_base = [c for c in candidatas if c in df.columns and not es_ingreso(c)]
if not {"pobre","anio"}.issubset(df.columns):
    raise KeyError("Faltan columnas obligatorias: pobre y/o anio")

# en este paso tipifico pobre y anio
df["pobre"] = pd.to_numeric(df["pobre"], errors="coerce")
df["anio"] = pd.to_numeric(df["anio"], errors="coerce")

# en este paso armo función de diseño: one-hot para categóricas, mantengo numéricas
def construir_X(subdf, cols):
    # detecto categóricas vs numéricas por dtype y cardinalidad
    cat_cols = []
    num_cols = []
    for c in cols:
        if pd.api.types.is_numeric_dtype(subdf[c]):
            num_cols.append(c)
        else:
            cat_cols.append(c)
    # dummies drop_first para evitar colinealidad
    X_cat = pd.get_dummies(subdf[cat_cols], drop_first=True, dtype=float) if cat_cols else pd.DataFrame(index=subdf.index)
    X_num = subdf[num_cols].apply(pd.to_numeric, errors="coerce")
    X = pd.concat([X_num, X_cat], axis=1)
    # agrego columna de unos para el intercepto al final
    X.insert(0, "constante", 1.0)
    return X

# en este paso separo por año
resp_2005 = df[df["anio"] == 2005].copy()
resp_2025 = df[df["anio"] == 2025].copy()

# en este paso construyo matrices X e y por año
X2005 = construir_X(resp_2005, vars_X_base)
y2005 = resp_2005["pobre"].astype(float)

X2025 = construir_X(resp_2025, vars_X_base)
y2025 = resp_2025["pobre"].astype(float)

# en este paso divido 70/30 con semilla 444
Xtr05, Xte05, ytr05, yte05 = train_test_split(X2005, y2005, test_size=0.3, random_state=444)
Xtr25, Xte25, ytr25, yte25 = train_test_split(X2025, y2025, test_size=0.3, random_state=444)

print(f"2005 → train {Xtr05.shape[0]}  test {Xte05.shape[0]}")
print(f"2025 → train {Xtr25.shape[0]}  test {Xte25.shape[0]}")

# en este paso creo función de diferencia de medias con test t Welch para cada feature
def tabla_diff_medias(X_train, X_test, excluir=("constante",)):
    cols_eval = [c for c in X_train.columns if c not in excluir]
    filas = []
    for c in cols_eval:
        a = pd.to_numeric(X_train[c], errors="coerce").dropna()
        b = pd.to_numeric(X_test[c],  errors="coerce").dropna()
        if len(a)==0 or len(b)==0:
            continue
        tstat, pval = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit")
        filas.append({
            "variable": c,
            "media_train": a.mean(),
            "media_test": b.mean(),
            "diferencia": a.mean() - b.mean(),
            "t_stat": tstat,
            "p_value": pval,
            "n_train": a.size,
            "n_test": b.size
        })
    out = pd.DataFrame(filas)
    # ordeno por p_value ascendente
    return out.sort_values("p_value").reset_index(drop=True)

# en este paso calculo tablas por año
diff05 = tabla_diff_medias(Xtr05, Xte05)
diff25 = tabla_diff_medias(Xtr25, Xte25)

# en este paso exporto a Excel
out05 = PATH_BASES / "tabla_diff_medias_train_vs_test_2005.xlsx"
out25 = PATH_BASES / "tabla_diff_medias_train_vs_test_2025.xlsx"
with pd.ExcelWriter(out05, engine="openpyxl") as w:
    diff05.to_excel(w, index=False, sheet_name="diff_medias_2005")
with pd.ExcelWriter(out25, engine="openpyxl") as w:
    diff25.to_excel(w, index=False, sheet_name="diff_medias_2025")

# en este paso muestro top filas para control rápido
print("\nTop variables con mayor evidencia de diferencia 2005")
print(diff05.head(10).to_string(index=False))
print("\nTop variables con mayor evidencia de diferencia 2025")
print(diff25.head(10).to_string(index=False))


In [ ]:
# PARTE A.2 - Separar bases respondieron y norespondieron por año
import pandas as pd
from pathlib import Path

# en este paso cargo las bases
PATH_BASES = Path.cwd()
file_resp = PATH_BASES / "respondieron_final.xlsx"
file_noresp = PATH_BASES / "norespondieron.xlsx"

respondieron = pd.read_excel(file_resp)
norespondieron = pd.read_excel(file_noresp)

# en este paso tipifico el año
respondieron["anio"] = pd.to_numeric(respondieron["anio"], errors="coerce")
norespondieron["anio"] = pd.to_numeric(norespondieron["anio"], errors="coerce")

# en este paso creo los subconjuntos por año
respondieron_2005 = respondieron[respondieron["anio"] == 2005].copy()
respondieron_2025 = respondieron[respondieron["anio"] == 2025].copy()

norespondieron_2005 = norespondieron[norespondieron["anio"] == 2005].copy()
norespondieron_2025 = norespondieron[norespondieron["anio"] == 2025].copy()

# en este paso muestro el tamaño de cada subconjunto
print("respondieron_2005:", respondieron_2005.shape)
print("respondieron_2025:", respondieron_2025.shape)
print("norespondieron_2005:", norespondieron_2005.shape)
print("norespondieron_2025:", norespondieron_2025.shape)

# en este paso guardo las bases separadas
respondieron_2005.to_excel(PATH_BASES / "respondieron_2005.xlsx", index=False)
respondieron_2025.to_excel(PATH_BASES / "respondieron_2025.xlsx", index=False)
norespondieron_2005.to_excel(PATH_BASES / "norespondieron_2005.xlsx", index=False)
norespondieron_2025.to_excel(PATH_BASES / "norespondieron_2025.xlsx", index=False)

print("Listo, se generaron las 4 bases por año.")



---

# Modelo de Regresion Logistica

* 3) Estimacion y efectos Marginales


In [ ]:
#Librerias a utilizar Punto B y D
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegressionCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

# Estimación del modelo
logit_sm = sm.Logit(ytr25, Xtr25) #Logit con las variables de entrenamiento de 2025
res_logit = logit_sm.fit(disp=0)

# Tabla de resultados con coef, std err y odds-ratio
tabla_logit = pd.DataFrame({
    "Variable": Xtr25.columns,
    "Coeficiente": res_logit.params,
    "Error Std.": res_logit.bse,
    "p-valor": res_logit.pvalues,
})
tabla_logit["Odds Ratio"] = np.exp(tabla_logit["Coeficiente"])

display(tabla_logit.round(4))

# Exportar a Excel
tabla_logit.to_excel("resultados_logit_2025.xlsx", index=False)

#Predicciones
y_prob_logit = res_logit.predict(Xte25)

#desempeño
y_pred_logit = (y_prob_logit >= 0.5).astype(int)
print(classification_report(yte25, y_pred_logit))


* 4) Visualización

In [ ]:
import seaborn as sns
# Selecciono una variable numérica relevante
var_num = "edad" if "edad" in Xte25.columns else "cantidad_miembros"

#  DataFrame con probabilidad predicha y la variable elegida
df_plot = pd.DataFrame({
    var_num: Xte25[var_num],
    "prob_pobre": y_prob_logit
})

# Gráfico el scatter mas la linea de tendencia
plt.figure(figsize=(8,5))
sns.scatterplot(data=df_plot, x=var_num, y="prob_pobre", alpha=0.3, label="Observaciones")
sns.regplot(data=df_plot, x=var_num, y="prob_pobre",
            lowess=True, scatter=False, color="red", label="Tendencia (LOWESS)")
plt.title(f"Probabilidad estimada de ser pobre según {var_num}")
plt.xlabel(var_num)
plt.ylabel(r"$\hat{P}(\text{pobre}=1 \mid X)$")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


---

# Método de Vecinos Cercanos (KNN)


* 5) Estimación

In [ ]:
#PARTE C.5

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# Aseguro entrada numérica (si hay categóricas, paso a dummies)
Xtr25_knn = pd.get_dummies(Xtr25, drop_first=True)
Xte25_knn = pd.get_dummies(Xte25, drop_first=True)
# Alineo columnas entre train y test
Xte25_knn = Xte25_knn.reindex(columns=Xtr25_knn.columns, fill_value=0)

resultados_knn = {}

for k in [1, 5, 10]:
    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),  # no centra, útil con dummies dispersas
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])
    pipe.fit(Xtr25_knn, ytr25)
    ypred = pipe.predict(Xte25_knn)
    resultados_knn[k] = {"modelo": pipe, "y_pred": ypred}

    print(f"\n=== KNN (K={k}) — desempeño en test 2025 ===")
    print(classification_report(yte25, ypred, digits=3))


* 6) Visualización

In [ ]:
#PARTE C.6

import matplotlib.pyplot as plt


# Reutilizo matrices del Ítem 5; si no existen (celda corrida en frío), las creo al vuelo
if "Xtr25_knn" not in globals():
    Xtr25_knn = pd.get_dummies(Xtr25, drop_first=True)
    Xte25_knn = pd.get_dummies(Xte25, drop_first=True)
    Xte25_knn = Xte25_knn.reindex(columns=Xtr25_knn.columns, fill_value=0)

# Si no tengo modelos K=1 y K=10 entrenados, los entreno rápido
if "resultados_knn" not in globals() or (1 not in resultados_knn or 10 not in resultados_knn):
    resultados_knn = {}
    for k in [1, 10]:
        pipe = Pipeline([
            ("scaler", StandardScaler(with_mean=False)),
            ("knn", KNeighborsClassifier(n_neighbors=k))
        ])
        pipe.fit(Xtr25_knn, ytr25)
        resultados_knn[k] = {"modelo": pipe}

# Selección de dos features “típicas” de la materia si existen; si no, primeras dos columnas
preferidas = [c for c in ["edad", "cantidad_miembros", "n_personas", "horastrab"] if c in Xtr25_knn.columns]
f1, f2 = (preferidas + list(Xtr25_knn.columns))[:2]

def plot_frontera(modelo, k):
    # Malla en el espacio (f1, f2)
    x_min, x_max = Xtr25_knn[f1].min() - 0.5, Xtr25_knn[f1].max() + 0.5
    y_min, y_max = Xtr25_knn[f2].min() - 0.5, Xtr25_knn[f2].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))

    # Armamos un DataFrame con TODAS las columnas de Xtr25_knn, seteo 0 salvo f1 y f2
    base = pd.DataFrame(0, index=np.arange(xx.size), columns=Xtr25_knn.columns)
    base[f1] = xx.ravel()
    base[f2] = yy.ravel()

    Z = modelo.predict(base).reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, Z, alpha=0.25, levels=2)
    plt.scatter(Xtr25_knn[f1], Xtr25_knn[f2], c=ytr25, s=15, alpha=0.6, edgecolor='none')
    plt.title(f"Frontera de decisión KNN (K={k}) — clases pobre/no pobre")
    plt.xlabel(f1); plt.ylabel(f2)
    plt.tight_layout()
    plt.show()

# Mostrar fronteras para K=1 y K=10
plot_frontera(resultados_knn[1]["modelo"], 1)
plot_frontera(resultados_knn[10]["modelo"], 10)


* 7) K optimo por Cross-validation

In [ ]:
#PARTE C.7

from sklearn.model_selection import cross_val_score


# Reuso matrices del Ítem 5; si no existen, las genero al vuelo
if "Xtr25_knn" not in globals():
    Xtr25_knn = pd.get_dummies(Xtr25, drop_first=True)
    Xte25_knn = pd.get_dummies(Xte25, drop_first=True)
    Xte25_knn = Xte25_knn.reindex(columns=Xtr25_knn.columns, fill_value=0)

ks = list(range(1, 10+1))
acc_mean, acc_std = [], []

for k in ks:
    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])
    scores = cross_val_score(pipe, Xtr25_knn, ytr25, cv=5, scoring="accuracy", n_jobs=-1)
    acc_mean.append(scores.mean())
    acc_std.append(scores.std())

# Tabla de resultados CV
cv_df = pd.DataFrame({"K": ks, "accuracy_mean": acc_mean, "accuracy_std": acc_std})
print("\n=== CV 5-fold — Accuracy por K ===")
display(cv_df.round(4))

# Selección de K óptimo (máximo accuracy promedio)
k_opt = cv_df.loc[cv_df["accuracy_mean"].idxmax(), "K"]
print(f"\nK óptimo (CV 5-fold, métrica accuracy): {int(k_opt)}")

# Entreno el modelo final KNN con K–CV en todo Xtr25
pipe_kcv = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("knn", KNeighborsClassifier(n_neighbors=int(k_opt)))
])
pipe_kcv.fit(Xtr25_knn, ytr25)

# Desempeño del KNN K–CV en test 2025
y_pred_kcv_test = pipe_kcv.predict(Xte25_knn)
print("\n=== KNN con K–CV — desempeño en test 2025 ===")
print(classification_report(yte25, y_pred_kcv_test, digits=3))

# (Opcional) Visual rápido en pantalla (no se guarda)
plt.figure(figsize=(7,5))
plt.plot(cv_df["K"], cv_df["accuracy_mean"], marker="o")
plt.fill_between(cv_df["K"],
                 cv_df["accuracy_mean"] - cv_df["accuracy_std"],
                 cv_df["accuracy_mean"] + cv_df["accuracy_std"],
                 alpha=0.2)
plt.xlabel("K"); plt.ylabel("Accuracy (CV 5-fold)")
plt.title("Selección de K por validación cruzada (respondieron_2025)")
plt.grid(True, alpha=0.3); plt.tight_layout()
plt.show()


# Modelo de Regresion Logistica con Regularización: Ridg y LASSO

* 8) Visualizacion

In [ ]:
# Definimos grilla de penalización: C = 1/λ
C_grid = np.logspace(-5, 5, 11)  # 11 puntos entre 10^-5 y 10^5

#Guardamos en un array los coeficientes
coefs_lasso = []
coefs_ridge = []

# Entrenamos modelos para cada valor de C y guardamos los coeficientes
for C in C_grid:
    # LASSO
    lasso = LogisticRegression(penalty='l1', solver='liblinear', C=C, random_state=444, max_iter=3000) #Ajustamos
    lasso.fit(Xtr25, ytr25) #entrenamos
    coefs_lasso.append(lasso.coef_.ravel()) # guardamos

    # RIDGE
    ridge = LogisticRegression(penalty='l2', solver='lbfgs', C=C, random_state=444, max_iter=3000)#Ajustamos
    ridge.fit(Xtr25, ytr25)# entrenamos
    coefs_ridge.append(ridge.coef_.ravel()) # guardamos

# Los guardamos en el array que armamos en un principio
coefs_lasso = np.array(coefs_lasso)
coefs_ridge = np.array(coefs_ridge)


#Plot
plt.figure(figsize=(12,5))

# ---- LASSO ----
plt.subplot(1,2,1)
for i, var in enumerate(Xtr25.columns):
    plt.plot(C_grid, coefs_lasso[:, i], label=var)
plt.xscale('log')
plt.title("Trayectoria de coeficientes - LASSO (L1)")
plt.xlabel("C (1/λ)")
plt.ylabel("Coeficientes")
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()

# ---- RIDGE ----
plt.subplot(1,2,2)
for i, var in enumerate(Xtr25.columns):
    plt.plot(C_grid, coefs_ridge[:, i], label=var)
plt.xscale('log')
plt.title("Trayectoria de coeficientes - Ridge (L2)")
plt.xlabel("C (1/λ)")
plt.ylabel("Coeficientes")
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()

plt.show()

* 9) Penalidad Optima por Cross-Validation y Visualizacion

In [ ]:
# Configuramos la penalización
C_grid = np.logspace(-5, 5, 11)
lambda_grid = 1 / C_grid

#Estrandarizamos datos
scaler = StandardScaler()
Xtr25_scaled = scaler.fit_transform(Xtr25)

#ENTRENAMIENTO CON LOGISTIC REGRESSION CV (5-FOLD)

# Modelo LASSO (L1)
lasso_cv = LogisticRegressionCV(Cs=C_grid, cv=5, penalty='l1', solver='liblinear', random_state=444, max_iter=5000, scoring='accuracy', n_jobs=-1)
lasso_cv.fit(Xtr25_scaled, ytr25)

# Modelo RIDGE (L2)
ridge_cv = LogisticRegressionCV(Cs=C_grid, cv=5, penalty='l2', solver='lbfgs', random_state=444, max_iter=5000, scoring='accuracy', n_jobs=-1)
ridge_cv.fit(Xtr25_scaled, ytr25)

#RESULTADOS DE λ ÓPTIMO
lambda_opt_lasso = 1 / lasso_cv.C_[0]
lambda_opt_ridge = 1 / ridge_cv.C_[0]

print(f"λ_opt LASSO (L1): {lambda_opt_lasso:.6f}")
print(f"λ_opt RIDGE (L2): {lambda_opt_ridge:.6f}")
print(f"C_opt LASSO: {lasso_cv.C_[0]:.6f}")
print(f"C_opt RIDGE: {ridge_cv.C_[0]:.6f}")

# 4. BOX-PLOTS DEL ERROR DE CLASIFICACIÓN

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Preparar datos para boxplots
error_lasso = 1 - lasso_cv.scores_[1]  # Error = 1 - Accuracy
error_ridge = 1 - ridge_cv.scores_[1]
positions = range(1, len(C_grid) + 1)
labels = [f'{lam:.1e}' for lam in lambda_grid]

#print(error_lasso)
#print(error_ridge)

# BOX-PLOT LASSO
box_lasso = ax1.boxplot([error_lasso[:, i] for i in range(len(C_grid))],
                       positions=positions, widths=0.7, patch_artist=True)
for patch in box_lasso['boxes']:
    patch.set_facecolor('lightblue')
    patch.set_alpha(0.7)
ax1.set_title('Distribución del Error de Clasificación - LASSO (L1)',
              fontsize=14, fontweight='bold', pad=20)
ax1.set_xlabel('λ (Parámetro de Regularización)', fontsize=12)
ax1.set_ylabel('Error de Clasificación (1 - Accuracy)', fontsize=12)
ax1.set_xticks(positions)
ax1.set_xticklabels(labels, rotation=45, ha='right')
opt_idx_lasso = np.argmin(np.abs(lambda_grid - lambda_opt_lasso))
ax1.axvline(x=positions[opt_idx_lasso], color='red', linestyle='--', linewidth=2,
           label=f'λ_opt = {lambda_opt_lasso:.2e}')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.set_ylim([0, max(error_lasso.max(), error_ridge.max()) + 0.05])

# BOX-PLOT RIDGE
box_ridge = ax2.boxplot([error_ridge[:, i] for i in range(len(C_grid))],
                       positions=positions, widths=0.7, patch_artist=True)
for patch in box_ridge['boxes']:
    patch.set_facecolor('lightgreen')
    patch.set_alpha(0.7)
ax2.set_title('Distribución del Error de Clasificación - RIDGE (L2)',
              fontsize=14, fontweight='bold', pad=20)
ax2.set_xlabel('λ (Parámetro de Regularización)', fontsize=12)
ax2.set_ylabel('Error de Clasificación (1 - Accuracy)', fontsize=12)
ax2.set_xticks(positions)
ax2.set_xticklabels(labels, rotation=45, ha='right')
opt_idx_ridge = np.argmin(np.abs(lambda_grid - lambda_opt_ridge))
ax2.axvline(x=positions[opt_idx_ridge], color='red', linestyle='--', linewidth=2,
           label=f'λ_opt = {lambda_opt_ridge:.2e}')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.set_ylim([0, max(error_lasso.max(), error_ridge.max()) + 0.05])

plt.tight_layout()
plt.show()

# BOX-PLOT OPCIONAL: PROPORCIÓN DE VARIABLES IGNORADAS (LASSO)
# Para mayor robustez, calcular sobre múltiples random states
random_states = [444, 445, 446, 447, 448]
prop_ceros_por_lambda = []

for i, C in enumerate(C_grid):
    prop_ceros_folds = []
    for rs in random_states:
        model = LogisticRegression(
            penalty='l1',
            solver='liblinear',
            C=C,
            random_state=rs,
            max_iter=5000
        )
        model.fit(Xtr25_scaled, ytr25)
        # Calcular proporción de coeficientes iguales a cero
        prop_cero = np.mean(model.coef_.ravel() == 0)
        prop_ceros_folds.append(prop_cero)

    prop_ceros_por_lambda.append(prop_ceros_folds)
    print(f"λ = {lambda_grid[i]:.1e}: Proporción media de vars ignoradas = {np.mean(prop_ceros_folds):.3f}")

# Gráfico de variables ignoradas
plt.figure(figsize=(14, 6))
box_ignoradas = plt.boxplot(prop_ceros_por_lambda, positions=positions, widths=0.7, patch_artist=True)

# Colorear cajas
for patch in box_ignoradas['boxes']:
    patch.set_facecolor('salmon')
    patch.set_alpha(0.7)
plt.title('Proporción de Variables Ignoradas - LASSO (L1)',
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('λ (Parámetro de Regularización)', fontsize=12)
plt.ylabel('Proporción de Coeficientes = 0', fontsize=12)
plt.xticks(positions, labels, rotation=45, ha='right')
plt.axvline(x=positions[opt_idx_lasso], color='red', linestyle='--', linewidth=2,
           )
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3, linestyle='--')
plt.ylim([-0.05, 1.05])

plt.tight_layout()
plt.show()



* 10) Comparacion de coeficientes

In [ ]:
# Recuperamos los valores óptimos de C
C_opt_lasso = lasso_cv.C_[0]
C_opt_ridge = ridge_cv.C_[0]

# Modelo sin penalidad
logit_simple = LogisticRegression(penalty=None, solver='lbfgs', max_iter=3000)
logit_simple.fit(Xtr25, ytr25)

# Modelo LASSO con penalidad óptima
logit_lasso = LogisticRegression(penalty='l1', solver='liblinear', C=C_opt_lasso, max_iter=3000)
logit_lasso.fit(Xtr25, ytr25)

# Modelo Ridge con penalidad óptima
logit_ridge = LogisticRegression(penalty='l2', solver='lbfgs', C=C_opt_ridge, max_iter=3000)
logit_ridge.fit(Xtr25, ytr25)

# Tabla comparativa
coef_df = pd.DataFrame({
    "Variable": Xtr25.columns,
    "Sin penalidad": logit_simple.coef_.ravel(),
    "LASSO (λ^cv)": logit_lasso.coef_.ravel(),
    "Ridge (λ^cv)": logit_ridge.coef_.ravel()
})

# Guardar tabla
coef_df.to_excel("coeficientes_logit_comparacion_2025.xlsx", index=False)

# Mostrar primeras filas
display(coef_df.head(10))


# Bonus : Elastic Net

In [ ]:
# Definir rango de C (inverso de lambda) y ratios L1
C_grid = np.logspace(-3, 2, 20)  # Ejemplo: de 0.001 a 100
l1_ratios = np.linspace(0.1, 0.9, 5)  # De 10% a 90% L1

# Entrenamiento
elastic_cv = LogisticRegressionCV(Cs=C_grid,cv=5,penalty='elasticnet',solver='saga',l1_ratios=l1_ratios,random_state=444,max_iter=3000,scoring='accuracy')
elastic_cv.fit(Xtr25, ytr25)

print(f"C óptimo Elastic Net: {elastic_cv.C_[0]:.4f}")
print(f"Ratio L1 óptimo: {elastic_cv.l1_ratio_[0]:.2f}")

# Tabla
coef_elastic = pd.DataFrame({
    "Variable": Xtr25.columns,
    "Coef_ElasticNet": elastic_cv.coef_.ravel()
})
display(coef_elastic.head(10))

In [ ]:
l1_ratios = np.linspace(0.1, 0.9, 5)
coefs_elastic = []

for ratio in l1_ratios:
    model = LogisticRegression(
        penalty='elasticnet', solver='saga',
        l1_ratio=ratio, C=elastic_cv.C_[0],
        random_state=444, max_iter=3000
    )
    model.fit(Xtr25, ytr25)
    coefs_elastic.append(model.coef_.ravel())

coefs_elastic = pd.DataFrame(coefs_elastic, columns=Xtr25.columns, index=[f"L1={r:.1f}" for r in l1_ratios])

plt.figure(figsize=(10,6))
sns.heatmap(coefs_elastic, cmap="coolwarm", center=0, annot=False)
plt.title("Elastic Net - Coeficientes según mezcla L1/L2 (l1_ratio)")
plt.xlabel("Variables")
plt.ylabel("Proporción L1")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
